> 在线讲义：[第 16 章 · 蒙特卡洛与采样：用样本平均代替积分，重要性采样、CEM 和 MPPI](https://wikieden.github.io/math-and-eval-stats/lessons/16-monte-carlo-sampling.html) · 本书《数学与实验统计》

# 第 16 章 · 蒙特卡洛与采样：用样本平均代替积分，重要性采样、CEM 和 MPPI

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/16-monte-carlo-sampling.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 MC | 样本平均 | `mc_mean_x2(n)`、`mc_error(n)` | n = 100：标准差 0.1413 |
| 第二步 IS | f·p/q | `tail_importance(1000, 3.0)`、`relative_error(...)` | 0.00124，相对误差 0.06 |
| 第三步 PPO | π_new/π_old | `ppo_ratio(0.30, 0.20)` | 1.5 |
| 第四步 CEM | 精英更新 | `cem()` | 第 4 轮 μ = 2.001 |
| 第四步 MPPI | exp(−c/λ) | `mppi_weights(c, 1.0)` | [0.477, 0.320, 0.194, 0.010] |

例子：估计 E[X²]（X ~ N(0, 1)，真值 1）；估计尾部概率 P(X > 3)（真值 0.00135）；PPO 的比率；一维 CEM 与 MPPI。
里程碑 ME-M12 的二维代价函数对比留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import numpy as np
from scipy.stats import norm
TRUE_TAIL = float(norm.sf(3.0))

**函数 `mc_mean_x2`**：普通蒙特卡洛：抽 n 个 X，平均 X²。

In [ ]:
def mc_mean_x2(n: int, seed: int = 0) -> float:
    """普通蒙特卡洛：抽 n 个 X，平均 X²。"""
    x = np.random.default_rng(seed).standard_normal(n)
    return float((x**2).mean())

**函数 `mc_error`**：重复 reps 次，看估计值的标准差（理论值 √(Var(X²)/n) = √(2/n)）。

In [ ]:
def mc_error(n: int, reps: int = 2000) -> float:
    """重复 reps 次，看估计值的标准差（理论值 √(Var(X²)/n) = √(2/n)）。"""
    rng = np.random.default_rng(1)
    return float((rng.standard_normal((reps, n)) ** 2).mean(1).std())

**函数 `tail_naive`**：直接数 X > 3 的比例。

In [ ]:
def tail_naive(n: int, seed: int = 0) -> float:
    """直接数 X > 3 的比例。"""
    return float((np.random.default_rng(seed).standard_normal(n) > 3).mean())

**函数 `tail_importance`**：重要性采样：从 N(shift, 1) 抽，权重 w = p(x)/q(x)，估计 E_q[1(x>3)·w]。shift = 0 就是直接数。

In [ ]:
def tail_importance(n: int, shift: float, seed: int = 0) -> float:
    """重要性采样：从 N(shift, 1) 抽，权重 w = p(x)/q(x)，估计 E_q[1(x>3)·w]。shift = 0 就是直接数。"""
    x = np.random.default_rng(seed).normal(shift, 1.0, n)
    w = np.exp(norm.logpdf(x) - norm.logpdf(x, shift))
    return float(((x > 3) * w).mean())

**函数 `relative_error`**：重复 reps 次，估计值的标准差 / 真值。

In [ ]:
def relative_error(n: int, shift: float, reps: int = 500) -> float:
    """重复 reps 次，估计值的标准差 / 真值。"""
    est = np.array([tail_importance(n, shift, seed=s) for s in range(reps)])
    return float(est.std() / TRUE_TAIL)

**函数 `ppo_ratio`**：PPO 的比率 r = π_new(a|s)/π_old(a|s)：就是用旧策略的样本估新策略期望时的重要性权重。

In [ ]:
def ppo_ratio(p_new: float, p_old: float) -> float:
    """PPO 的比率 r = π_new(a|s)/π_old(a|s)：就是用旧策略的样本估新策略期望时的重要性权重。"""
    return p_new / p_old

**函数 `cost`**：一维代价：(x − 2)²，最小点 x = 2。

In [ ]:
def cost(x: np.ndarray) -> np.ndarray:
    """一维代价：(x − 2)²，最小点 x = 2。"""
    return (x - 2.0) ** 2

**函数 `cem`**：交叉熵方法：从 N(μ, σ²) 抽 n 个，留代价最低的 elite 个，用它们的均值和标准差更新。返回每轮 (μ, σ)。

In [ ]:
def cem(iters: int = 6, n: int = 50, elite: int = 10, seed: int = 0) -> list[tuple[float, float]]:
    """交叉熵方法：从 N(μ, σ²) 抽 n 个，留代价最低的 elite 个，用它们的均值和标准差更新。返回每轮 (μ, σ)。"""
    rng = np.random.default_rng(seed)
    mu, sd, hist = 0.0, 2.0, [(0.0, 2.0)]
    for _ in range(iters):
        x = rng.normal(mu, sd, n)
        e = x[np.argsort(cost(x))[:elite]]
        mu, sd = float(e.mean()), float(e.std() + 1e-6)
        hist.append((mu, sd))
    return hist

**函数 `mppi_weights`**：MPPI 的指数加权：w ∝ exp(−(c − c_min)/λ)，λ 越小越只信最好的那几个。

In [ ]:
def mppi_weights(costs: np.ndarray, lam: float) -> np.ndarray:
    """MPPI 的指数加权：w ∝ exp(−(c − c_min)/λ)，λ 越小越只信最好的那几个。"""
    w = np.exp(-(costs - costs.min()) / lam)
    return w / w.sum()

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 蒙特卡洛：用样本平均代替积分，误差 ∝ 1/√n")
for n in (100, 10000):
    print(f"  n = {n:5d}：E[X²] ≈ {mc_mean_x2(n):.4f}；重复 2000 次的标准差 {mc_error(n):.4f}（理论 {math.sqrt(2 / n):.4f}）")

In [ ]:
print("\n第二步 重要性采样：估计罕见事件 P(X > 3)")
print(f"  真值 {TRUE_TAIL:.5f}")
for shift, name in ((0.0, "直接数（提议 = 原分布 N(0, 1)）"), (3.0, "提议 N(3, 1)：样本集中在尾部"), (-1.0, "提议 N(−1, 1)：偏到反方向")):
    print(f"  {name}：1000 个样本估计 {tail_importance(1000, shift):.5f}，相对误差 {relative_error(1000, shift):.2f}")

In [ ]:
print("\n第三步 PPO 的比率就是重要性权重")
print(f"  旧策略选这个动作的概率 0.20，新策略 0.30：r = {ppo_ratio(0.30, 0.20):.2f}；截断到 [0.8, 1.2] 后用 1.2")

In [ ]:
print("\n第四步 CEM：抽样 → 留精英 → 更新分布")
for k, (mu, sd) in enumerate(cem()):
    print(f"  第 {k} 轮：μ = {mu:.3f}，σ = {sd:.3f}")

In [ ]:
print("\n第五步 MPPI：按代价做指数加权，λ 是温度")
c = np.array([0.1, 0.5, 1.0, 4.0])
for lam in (0.1, 1.0, 10.0):
    print(f"  代价 {c.tolist()}，λ = {lam}：权重 {np.round(mppi_weights(c, lam), 3).tolist()}")

## 3. 代码里才看得到的

1. **权重在对数空间算**：p、q 都很小时直接相除会下溢成 0/0；先算 log p − log q 再取 exp。
2. **MPPI 先减最小代价**：代价都很大时 exp(−c/λ) 全部下溢成 0，归一化就变成 0/0；减掉 c_min 不改变归一化后的权重（第 22 章 logsumexp 同一个技巧）。
3. **CEM 的 σ 会塌到 0**：精英都挤在一起时标准差趋于 0，再也探索不了；实际实现会给 σ 加一个下限或噪声，代码里只加了 1e-6 防止除零。

## 4. 自己改一改

1. 把重要性采样的提议分布换成 N(3, 0.3²)（太窄），相对误差变好还是变坏？
2. 把 CEM 的精英数从 10 改成 2，收敛更快还是更容易卡住？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 扔石子估湖面，误差根号 n 分之一，和维度无关；
> 去山顶抽人再换算，权重 p 比 q，提议分布要盖住关键处；
> PPO 的比率是权重，截断防止爆方差；
> CEM 留精英，MPPI 按 e 的负代价除温度加权。